# Authority and promotion

<a id="keep-authority-where-it-belongs"></a>

What stops a proposed change from changing the test instead of improving the
answer? A declared mutation boundary can reject an attempt to change what a
candidate is allowed to edit. The evaluator still judges permitted revisions,
including revisions that make the answers worse.

We'll reuse the [help assistant](https://sentient-xyz.github.io/meta-evolve-docs/guides/harness-evolution/): change its
matching instruction, ask three questions, and check the answers. This time,
a structured harness declares the editable fields. One revision improves the
answers, one regresses, and one tries to redefine the planner surface as
`evaluation_checks`. The last is refused before evaluation.

The worker and proposer are handwritten Python simulations. This lesson runs
in-process with no model, keys, or earlier notebook state. Its validation is
**not a security sandbox for arbitrary Python**.



## 1. Install

Use a fresh notebook environment running **Python 3.12 or newer**.
Install directly from the published documentation:

In [ ]:
%pip install https://sentient-xyz.github.io/meta-evolve-docs/downloads/meta-evolve.zip

If you already imported Meta-Evolve, restart the kernel after installing.
Then run the remaining cells in order.

**Archived or offline docs:** use the ZIP included with that build. Put
`meta-evolve.zip` in the notebook's working folder (`%pwd` shows it; hosted
notebooks let you upload files), then run `%pip install ./meta-evolve.zip`
instead. Installing from source may still download build tools.

## 2. Keep the worker and expected answers fixed

Exact matching answers “reset password” but misses “please reset password.”
Topic matching allows extra words around the same topic. The worker recognizes
these two instructions; it does not interpret arbitrary prompts.

`HELP_NOTES` supplies its replies. `HELP_CASES` defines the fixed questions and
expected answers. Neither belongs to the configuration being revised.

In [ ]:
from dataclasses import replace

import meta_evolve as meta
from meta_evolve import harness
from meta_evolve.domain import ComponentRef

HELP_NOTES = (
    ("reset password", "Open Settings > Password."),
    ("download invoice", "Open Billing > Invoices."),
)
HELP_CASES = (
    ("reset password", "Open Settings > Password."),
    ("please reset password", "Open Settings > Password."),
    ("download invoice", "Open Billing > Invoices."),
)


def answer(question, configuration):
    instruction = configuration.planner.value
    if instruction not in ("Match exact questions.", "Match topic words."):
        raise ValueError("Unsupported matching instruction")
    question = question.lower()
    for topic, reply in HELP_NOTES:
        if instruction == "Match exact questions.":
            matches = question == topic
        else:
            matches = set(topic.split()) <= set(question.split())
        if matches:
            return reply
    return "I don't have a matching help note."

## 3. Declare the editable values and evaluate real answers

`HarnessArtifact` has two mutable values: planner text and context-policy
configuration. This example changes only the planner text. The parameter names,
component references, and interfaces are fixed declarations; the context value
is not used by this small worker.

The evaluator asks all three questions and retains the actual answer, expected
answer, and pass/fail result. A missed question receives an ordinary poor score.
The score does not depend on a revision's label.

In [ ]:
HARNESS_SEED = harness.HarnessArtifact(
    planner=harness.TextParam(
        "planner", ComponentRef("planner", "help-matcher", "1"),
        "matching-instruction/v1", "Match exact questions.",
    ),
    context_policy=harness.PolicyParam(
        "context_policy", ComponentRef("context-policy", "fixed-notes", "1"),
        "help-context/v1", {"include": ("help-notes",)},
    ),
)


def evaluate_harness(configuration):
    checks = []
    for question, expected in HELP_CASES:
        actual = answer(question, configuration)
        checks.append({"question": question, "expected": expected,
                       "actual": actual, "passed": actual == expected})
    return meta.EvaluationResult(
        metrics={"score": sum(check["passed"] for check in checks) / len(checks)},
        evidence=(meta.EvidenceDraft(kind="help-checks", data={"checks": checks}),),
    )

## 4. Propose an improvement, a regression, and a forbidden edit

The first revision changes the planner's **value** to topic matching. The second
restores exact matching, so the polite password question fails again.

The third uses `replace` to rename the planner's mutation surface to `evaluation_checks`
and propose an always-pass rule. It bypasses the convenient mutation helper
deliberately. It attempts to change the declaration of what may be edited;
it never obtains or changes the actual evaluator or `HELP_CASES`.

In [ ]:
def allowed_improvement(parent):
    return parent.apply(harness.HarnessMutation("planner", "Match topic words."))


def allowed_regression(parent):
    return parent.apply(harness.HarnessMutation("planner", "Match exact questions."))


def redefine_checks(parent):
    return replace(parent, planner=replace(
        parent.planner, name="evaluation_checks", value="accept every answer",
    ))


def propose_harness(parent):
    return next(proposal_steps)(parent)


def show_outcome(label, trial):
    work = f"attempts +{trial.usage.trials}, evaluations +{trial.usage.evaluations}"
    if trial.failure:
        print(f"{label}: {trial.failure.kind}; {work}")
        print("Refusal record: authority_result.trials()[-1]")
        print("Reason:", trial.failure.message, dict(trial.failure.details))
    else:
        checks = trial.evidence[0].data["checks"]
        print(f"{label}: {sum(c['passed'] for c in checks)}/{len(checks)} checks; {work}")
        check = checks[1]
        print(check["question"], "->", check["actual"], check["passed"])

## 5. Run and inspect the retained outcomes

The expanded `Task` declares the harness artifact type; `improve()` uses the
default artifact type and cannot select this structured codec. The allowance
covers three proposal attempts and up to four evaluations, including the seed.
There is no score target that would stop the run at the first perfect result.

In [ ]:
proposal_steps = iter((allowed_improvement, allowed_regression, redefine_checks))
authority_task = meta.Task(
    artifact=harness.HarnessArtifact, evaluator=evaluate_harness,
    objectives=(meta.Maximize("score"),), budget=meta.Budget(trials=3, evaluations=4),
)
authority_experiment = meta.Experiment(
    task=authority_task, seed=HARNESS_SEED, proposer=propose_harness,
    search=meta.Greedy(max_trials=3),
)
authority_result = meta.run(authority_experiment)
for label, trial in zip(("Starting", "Improvement", "Regression", "Attempt 3"),
                        authority_result.trials()):
    show_outcome(label, trial)
selected_harness = authority_result.best().value
print("Original declaration:", HARNESS_SEED.planner.name)
print("Protected polite check:", HELP_CASES[1])
print("Selected instruction:", selected_harness.planner.value)
work = authority_result.usage()
print("Total:", work.trials, "attempts;", work.evaluations, "evaluations")
# Output:
# Starting: 2/3 checks; attempts +0, evaluations +1
# please reset password -> I don't have a matching help note. False
# Improvement: 3/3 checks; attempts +1, evaluations +1
# please reset password -> Open Settings > Password. True
# Regression: 2/3 checks; attempts +1, evaluations +1
# please reset password -> I don't have a matching help note. False
# Attempt 3: policy_violation; attempts +1, evaluations +0
# Refusal record: authority_result.trials()[-1]
# Reason: harness successor expands mutation authority {'changed_declarations': ('planner.name',)}
# Original declaration: planner
# Protected polite check: ('please reset password', 'Open Settings > Password.')
# Selected instruction: Match topic words.
# Total: 3 attempts; 3 evaluations

The starting instruction passes **2/3** checks, the improved instruction passes
**3/3**, and the permitted regression passes **2/3**. Greedy keeps the better
instruction as its parent for the remaining proposals and as the selected result.
The regression is a distinct recorded version even though its content matches
the seed.

The final attempt has a typed **`PolicyViolation`**, a refusal reason, and no
score, successor artifact, or evaluation. Its record remains available through
`authority_result.trials()[-1]`; its `.id` identifies that retained attempt.
The output shows three attempts and three
evaluations: seed, improvement, and regression. The refused attempt spends one
proposal attempt and **zero evaluations**, leaving one evaluation unused.
Each evaluation asks three questions; those are checks within one evaluation.

### What enforces the refusal?

The built-in harness codec's `validate_successor` checks the encoded proposal
against its parent **before a successor artifact is created or evaluated**.
It finds the changed declaration `planner.name` and records `PolicyViolation`.
The experiment's checks, evaluator, objective, and budget remain unchanged.
The [public contract](https://sentient-xyz.github.io/meta-evolve-docs/programming-model/#authority-and-failure-boundary)
defines this boundary.

`HarnessArtifact.apply` also validates mutation requests, but that helper is
only a convenience. Asking it directly for an undeclared `evaluation_checks`
surface would return `InvalidOutput`. The demonstrated `replace` bypass shows
that core admission independently rejects the changed declaration.

This protects the structured proposal path. Ordinary in-process Python can
access notebook globals; declaring an independent evaluator does not isolate
malicious callbacks. No process or filesystem confinement is used here. A
path-valid `SourceTree` alone also does not enforce a file allowlist. For source
execution controls and their limits, see the
[source execution guide](https://sentient-xyz.github.io/meta-evolve-docs/guides/live-parser/#inspect-the-grader).

Preventing this declared forbidden edit does not prevent every way to exploit
an imperfect metric. Our three checks cannot establish general answer quality;
see the broader [specification-gaming research](https://sentient-xyz.github.io/meta-evolve-docs/research/specification-gaming/).

## Change and predict

In the final cell's `proposal_steps`, replace `redefine_checks` with
`allowed_regression`, then rerun that cell. Predict **four evaluations** and a
third revision with a real **2/3** score. The best instruction is still topic
matching, so restoring exact matching is a change to that parent each time.
Restore the forbidden edit to see the typed refusal again.

You can also change an expected answer as the experiment author and rerun.
The score should change with the checks. That is a new evaluation setup, not
authority a proposal receives within the original run.

## Three fields, three owners

**Authority** determines who may change a candidate, judge its results, or
approve its use. The proposer supplies a revision, the evaluator measures it,
search decides what to try or select, and governance controls promotion.

A trial's record splits into three kinds of fields:

- **Candidate-controlled** — the proposed artifact content and its declared
  citations. Everything a proposer decides, and nothing else.
- **Experiment-controlled** — the task, evaluator, hidden data, environment,
  seeds, and budgets. The evaluator scores independently; a proposer's claim
  about its own quality never moves a metric.
- **Governance-controlled** — permissions, admission, visibility, and
  promotion. These live outside the run entirely.

```text
candidate            experiment             governance
─────────            ──────────             ──────────
artifact content     evaluator + data       permissions
declared citations   budgets + seeds        visibility
                     environments           promotion
```

The experiment defines which fields a proposal may change. Evaluation rules,
budgets, recorded failures, and permissions are outside that mutation boundary.
The same rule applies when [a procedure becomes the candidate](https://sentient-xyz.github.io/meta-evolve-docs/concepts/meta-evolution/):
its permitted settings can change while the enclosing controls stay fixed.

Audited source inspection enforces grants on reads through the experience
interface. Verification and presentation stay outside candidate authority. See
[audited pull](https://sentient-xyz.github.io/meta-evolve-docs/guides/experience/#audited-pull) for access checks and limits.

## Proposal is not promotion

Search may select a best measured artifact. That selection grants no
production authority: **promotion** — actually deploying the selected
artifact — is an external governance act that no candidate and no search
policy can perform. This run leaves a selected instruction for the caller to
review and use. The advanced [harness repair study](https://sentient-xyz.github.io/meta-evolve-docs/research/meta-harness/)
also makes the boundary explicit with `promotion=external`; selection and
export do not deploy the result.

The authority invariants are canonical in the
[architecture](https://sentient-xyz.github.io/meta-evolve-docs/architecture/#authority-boundaries) and are not restated
here.

**Try it:** [Improve an agent's harness](https://sentient-xyz.github.io/meta-evolve-docs/guides/harness-evolution/).
**Related:** [Evaluation and evidence](https://sentient-xyz.github.io/meta-evolve-docs/concepts/evidence/) and [meta-evolution](https://sentient-xyz.github.io/meta-evolve-docs/concepts/meta-evolution/).
**Reference:** [Authority boundaries](https://sentient-xyz.github.io/meta-evolve-docs/architecture/#authority-boundaries).